In [1]:
import scipy.sparse as scs
import pandas as pd
import numpy as np

import scanpy as sc
import anndata
from sklearn.cluster import SpectralClustering
import pandas as pd

from scipy.sparse import csr_matrix

import anndata
import numpy as np
import scanpy as sc
import pandas as pd
from sklearn.metrics.cluster import contingency_matrix
from scipy.optimize import linear_sum_assignment
import scipy.sparse as scs


In [ ]:



def get_edge_overlap(grn_adata_sub, net, top_n = 150):
    result = {'top_n': top_n}
    result['modules'] = {}
    for g in grn_adata_sub.obs.grn.unique():
        subet = net[(net.grn == g) & (net.grn_effect>0.01)]
        subet['edge' ] = subet['source']+'_'+subet['target']
        subet['edge2' ] = subet['target']+'_'+subet['source']
        augmented = build_augmented_network(subet)

        module_overlap = []
        correct_orientation = []
        false_orientation = []

        for tn in range(10, top_n, 10):
            top_idx = get_top_genes(grn_adata_sub, g, top_n=tn)
            
            module_overlap.append(len(set(augmented.edge).intersection(set(grn_adata_sub.var.index[top_idx].tolist()))))
            correct_orientation.append(len(set(subet.edge).intersection(set(grn_adata_sub.var.index[top_idx].tolist()))))
            false_orientation.append(len(set(subet.edge2).intersection(set(grn_adata_sub.var.index[top_idx].tolist()))))



        dicto = {'correct_orientation_count_not_perturbed': correct_orientation,
                  'false_orientation_count_not_perturbed':false_orientation,
                  'module_overlap_not_perturbed': module_overlap}  
        result['modules'][str(g)] = pd.DataFrame(dicto)
        
    result = {'all_edges': result}
    return result


def get_edge_overlap_perturbed_only(grn_adata_sub, net, perturbed_regulators, top_n = 150):
    result = {'top_n': top_n}
    result['modules'] = {}
    for g in grn_adata_sub.obs.grn.unique():
        
        ## Check only the edges that are perturbed.
        subet = net[(net.source.isin(perturbed_regulators[perturbed_regulators.grn==g].disregulated_gene)) & (net.grn == g)]
        subet['edge' ] = subet['source']+'_'+subet['target']
        subet['edge2' ] = subet['target']+'_'+subet['source']
        augmented = build_augmented_network(subet)

        module_overlap = []
        correct_orientation = []
        false_orientation = []

        for tn in range(10, top_n, 10):
            top_idx = get_top_genes(grn_adata_sub, g, top_n=tn)
            
            module_overlap.append(len(set(augmented.edge).intersection(senetmapt(grn_adata_sub.var.index[top_idx].tolist()))))
            correct_orientation.append(len(set(subet.edge).intersection(set(grn_adata_sub.var.index[top_idx].tolist()))))
            false_orientation.append(len(set(subet.edge2).intersection(set(grn_adata_sub.var.index[top_idx].tolist()))))
            
        # Check the edges that are not perturbed
        subet2 = net[(net.grn == g) & (net.grn_effect>0.01)]

        subet2['edge' ] = subet2['source']+'_'+subet2['target']
        subet2['edge2' ] = subet2['target']+'_'+subet2['source']
        augmented2 = build_augmented_network(subet2)

        module_overlap_np = []
        correct_orientation_np = []
        false_orientation_np = []

        for tn in range(10, top_n, 10):
            top_idx = get_top_genes(grn_adata_sub, g, top_n=tn)
            
            module_overlap_np.append(len(set(augmented2.edge).intersection(set(grn_adata_sub.var.index[top_idx].tolist()))))
            correct_orientation_np.append(len(set(subet2.edge).intersection(set(grn_adata_sub.var.index[top_idx].tolist()))))
            false_orientation_np.append(len(set(subet2.edge2).intersection(set(grn_adata_sub.var.index[top_idx].tolist()))))
        
        
        # Check the edges that should not be there
        subet3 = net[(net.grn == g) & (net.grn_effect<=0.01)]

        subet3['edge' ] = subet3['source']+'_'+subet3['target']
        subet3['edge2' ] = subet3['target']+'_'+subet3['source']
        augmented3 = build_augmented_network(subet3)

        module_overlap_fp = []
        correct_orientation_fp = []
        false_orientation_fp = []

        for tn in range(10, top_n, 10):
            top_idx = get_top_genes(grn_adata_sub, g, top_n=tn)
            
            module_overlap_fp.append(len(set(augmented3.edge).intersection(set(grn_adata_sub.var.index[top_idx].tolist()))))
            correct_orientation_fp.append(len(set(subet3.edge).intersection(set(grn_adata_sub.var.index[top_idx].tolist()))))
            false_orientation_fp.append(len(set(subet3.edge2).intersection(set(grn_adata_sub.var.index[top_idx].tolist()))))
        
        
        dicto_df = {'correct_orientation_count': correct_orientation,
                  'false_orientation_count':false_orientation,
                  'module_overlap': module_overlap,
                  'correct_orientation_count_not_perturbed': correct_orientation_np,
                  'module_overlap_not_perturbed': module_overlap_np,
                  'false_orienfrom scipy.sparse import csr_matrixtation_count_not_perturbed': false_orientation_np,
                  'correct_orientation_count_fp': correct_orientation_fp,
                  'module_overlap_fp': module_overlap_fp,
                  'false_orientation_fp':false_orientation_fp
                  }
          
        dicto_df = pd.DataFrame(dicto_df)
        dicto_df['p_correct_orientation_count'] = dicto_df['correct_orientation_count']/subet.shape[0]
        dicto_df['p_false_orientation_count'] = dicto_df['false_orientation_count']/subet.shape[0]
        dicto_df['p_correct_orientation_count_not_perturbed'] = dicto_df['correct_orientation_count_not_perturbed']/subet2.shape[0]
        dicto_df['p_false_orientation_count_not_perturbed'] = dicto_df['false_orientation_count_not_perturbed']/subet2.shape[0]
        dicto_df['p_correct_orientation_count_fp'] = dicto_df['correct_orientation_count_fp']/subet3.shape[0]
        dicto_df['p_false_orientation_fp'] = dicto_df['false_orientation_fp']/subet3.shape[0]
        dicto_df['p_module_overlap'] = dicto_df['module_overlap']/augmented.shape[0]
        dicto_df['p_module_overlap_not_perturbed'] = dicto_df['module_overlap_not_perturbed']/augmented2.shape[0]
        dicto_df['p_module_overlap_fp'] = dicto_df['module_overlap_fp']/augmented3.shape[0]

        

        dicto = {'result_df': dicto_df,
                  'perturbed_edges':subet.shape[0],
                  'augmented_edges': augmented.shape[0],
                  'non_perturbed_edges': subet2.shape[0],
                  'augmented_edges_not_perturbed': augmented2.shape[0],
                  'fp_edges': subet3.shape[0],
                  'fp_augmented': augmented3.shape[0]
                  }  
        

        result['modules'][str(g)] = dicto
    
    
    result = {'perturbed_only': result}
    return result
    


def get_edgelist(grn_adata_sub, grn_varname = 'MAPK1_NegCtrl0', cluster_var='spectral_remap', top_n  = 150):
    top_idx = get_top_genes(grn_adata_sub, grn_varname, cluster_var=cluster_var, top_n=top_n)
    edgelist = grn_adata_sub.var.iloc[top_idx]
    edgelist = edgelist.reset_index()
    edgelist[['source', 'target']] = edgelist['index'].str.split('_', expand = True)
    return edgelist





def analysis(aac, nlc, net, top_genes=500):
    # Check if aac is sparse; convert to dense if necessary for operations like nansum
    if isinstance(aac, csr_matrix):
        sumi = np.array(aac.sum(axis=0)).flatten()  # Efficient summation for sparse
    else:
        sumi = np.array(np.nansum(aac, axis=0)).flatten()  # Fallback for dense
    
    sumia = np.flip(np.argsort(sumi))
    nlc = np.array(nlc)
    nlc = pd.DataFrame(nlc)
    nlc.columns = ['source', 'target']
    
    fraction_of_regulators_after_edge_selection = []
    for th in range(10, top_genes, 10):
        high = nlc.iloc[sumia[0:th]]
        high = pd.DataFrame(high)
        high.columns = ['source', 'target']
        rol = overrepresentation(edgelist=high)
        fraction_of_regulators_after_edge_selection.append(rol[rol.index.isin(net.source.unique())]['percentage'].sum())
    
    return fraction_of_regulators_after_edge_selection


    
def overrepresentation(edgelist, source = 'source', target = 'target'):
    occ = edgelist[source].value_counts().add(edgelist[target].value_counts(), fill_value=0)
    occ = pd.DataFrame(occ)
    occ = occ.sort_values('count', ascending=False)
    occ['percentage'] = occ['count']/occ['count'].sum()
    return occ


In [ ]:



def random_score_distributions(grn_adata, percentile = 95, column_of_interest = 'spectral', rel_abs = False):
    """
    Create random score distributions across clusters via subsampling
    Compare the score of the edges within one cluster and select only
    those which fill above the p percentile of the randomized scores.
    Return the indices of those edges.
    
    """
    if rel_abs:
        grn_adata.X = np.abs(grn_adata.X)
    relevant_indices = {}
    
    for c in grn_adata.obs[column_of_interest].unique():
        background_sum_distribution = []
        for i in range(1000):
            samples = len(np.where(grn_adata.obs[column_of_interest] == c)[0])
            rs = np.random.choice( range(grn_adata.obs.shape[0]),size=samples, replace=False)
            sum_of_lrps = grn_adata.X[rs, :].sum(axis = 0)
            background_sum_distribution.append(sum_of_lrps)
        background_sum_distribution = np.stack(background_sum_distribution)
        background_sum_distribution = np.array(background_sum_distribution)
        percentiles = np.percentile(background_sum_distribution, axis = 0, q = percentile)
    
        sum_of_interest = grn_adata.X[grn_adata.obs[column_of_interest] == c, :].sum(axis = 0)
        #sum_of_interest = np.array(sum_of_interest).reshape(sum_of_interest.shape[1])
        relevant_indices[c] = np.where((sum_of_interest-percentiles)>0)[0]
    return relevant_indices






def create_cluster_subset(grn_adata, grn_adata_sub, rel, cluster=0, obs = 'spectral', add_obs = ['spectral_remap']):
    ## subset only edges relevant in this cluster from original object
    sub1 = grn_adata[:, rel[int(cluster)]]
    #add the metadata from the remapped subset
    sub1.obs[[obs]+add_obs] = grn_adata_sub.obs[[obs]+add_obs] 
    # subset the correct group
    sub1 = sub1[sub1.obs[obs]==cluster]
    return sub1


def get_edges(sub1):
    sub1.var[['source','target']] =[x.split('_') for x in sub1.var.index.tolist() ]
    edges = scs.find(sub1.X)
    barcode = [sub1.obs.index[i] for i in edges[0]]
    source = [sub1.var.source[i] for i in edges[1]]
    target = [sub1.var.target[i] for i in edges[1]]
    edge = [sub1.var.index[i] for i in edges[1]]
    edgedf = pd.DataFrame({'barcode': barcode, 'edge': edge, 'source': source, 'target': target,  'value': edges[2]})
    
    summary = edgedf.groupby('edge').median('value')
    summary['mean'] = edgedf.groupby('edge').mean('value')
    summary = summary.rename(columns= {'value': 'median'})
    summary = summary.sort_values('median', ascending=False)
    return summary

def create_all_summaries(grn_adata, grn_adata_sub, rel, cluster_col = 'spectral'):
    summaries = []
    for c in np.unique(grn_adata_sub.obs[cluster_col]):
        sub1 = create_cluster_subset(grn_adata, grn_adata_sub, rel, cluster = c)
        summary = get_edges(sub1)
        summary[cluster_col] = c
        summaries.append(summary)
    summaries = pd.concat(summaries)
    return summaries



In [4]:
def split_index(aa):    
    aa.var['source']   = [l[0] for l in aa.var.index.str.split('_', expand=True)]
    aa.var['target']   = [l[1] for l in aa.var.index.str.split('_', expand=True)]
    return aa

In [5]:
scgenerai = sc.read_h5ad('/data_nfs/og86asub/netmap/netmap-evaluation/results/scgenerai/config/config_easy/net_51_43266_net_82_42088_net_105_43582/grn_lrp.h5ad')
csnet = sc.read_h5ad('/data_nfs/og86asub/netmap/netmap-evaluation/results/csnet/config/config_easy/net_51_43266_net_82_42088_net_105_43582/csnet.csn.h5ad')
netmap = sc.read_h5ad('/data_nfs/og86asub/netmap/netmap-evaluation/results/netmap/config/config_easy/net_51_43266_net_82_42088_net_105_43582/grn_lrp.h5ad')

/data_nfs/og86asub/netmap/netmap-evaluation/netmap/.pixi/envs/default/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [6]:
csnet = csnet[:, csnet.layers['significance'].sum(axis = 0)!= 0].copy()
csnet.var = csnet.var.set_index('edge')
csnet = split_index(csnet)

In [7]:
scgenerai.var

,source,target
ACTR6_ACADVL,ACTR6,ACADVL
ADA_ACADVL,ADA,ACADVL
ADD1_ACADVL,ADD1,ACADVL
ADIPOQ_ACADVL,ADIPOQ,ACADVL
AEBP1_ACADVL,AEBP1,ACADVL
...,...,...
hsa-miR-342_ZNF507,hsa-miR-342,ZNF507
hsa-miR-661_ZNF507,hsa-miR-661,ZNF507
ZW10_ZNF507,ZW10,ZNF507
hsa-miR-342_ZW10,hsa-miR-342,ZW10


In [8]:
scgenerai = split_index(scgenerai)

In [9]:
net = pd.read_csv('/data_nfs/og86asub/netmap/netmap-evaluation/data/clustered_network/net_82_42088/edges.tsv', sep= '\t')

In [10]:
net2 = pd.read_csv('/data_nfs/og86asub/netmap/netmap-evaluation/data/clustered_network/net_105_43582/edges.tsv', sep= '\t')

{'input_data': '/data_nfs/og86asub/netmap/netmap-evaluation/data/simulated_data/config_easy/net_105_43582_net_51_43266_net_82_42088/data.h5ad', 'output_directory': '/data_nfs/og86asub/netmap/netmap-evaluation/results/netmap/config/config_easy/net_105_43582_net_51_43266_net_82_42088', 'transcription_factors': '/data_nfs/datasets/SCENIC_DB/tf_lists/allTFs_hg38.txt', 'tf_only': False, 'penalize_error': True, 'adata_filename': 'grn_lrp.h5ad', 'grn': 'grn_lrp.tsv', 'masking_percentage': 0.1, 'masking_value': 0, 'print_every': 100, 'optimizer': 'Adam', 'learning_rate': 0.005, 'epochs': 150, 'overwrite': True, 'n_models': 10, 'n_top_edges': 100, 'test_size': 0.3, 'edge_count': 10000}


scgenerai


/tmp/ipykernel_488653/1577643170.py:123: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults please pass: flavor="igraph" and n_iterations=2.  directed must also be False to work with igraph's implementation.
  sc.tl.leiden(adata, resolution=config['leiden_resolution'])


netmap


/data_nfs/og86asub/netmap/netmap-evaluation/netmap/.pixi/envs/default/lib/python3.12/site-packages/sklearn/manifold/_spectral_embedding.py:329: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(
/data_nfs/og86asub/netmap/netmap-evaluation/netmap/.pixi/envs/default/lib/python3.12/site-packages/scipy/sparse/_index.py:210: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_arrayXarray(i, j, x)


csnet


/data_nfs/og86asub/netmap/netmap-evaluation/netmap/.pixi/envs/default/lib/python3.12/site-packages/scipy/sparse/_index.py:210: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_arrayXarray(i, j, x)
/data_nfs/og86asub/netmap/netmap-evaluation/netmap/.pixi/envs/default/lib/python3.12/site-packages/sklearn/manifold/_spectral_embedding.py:329: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


In [17]:
collect_results


{'scgenerai': {'net_105_43582': {'edge_overlap': 39,
   'net_size': 110,
   'overlap_percent': 0.35454545454545455},
  'total_number_edges': 28203,
  'net_51_43266': {'edge_overlap': 68,
   'net_size': 76,
   'overlap_percent': 0.8947368421052632},
  'net_82_42088': {'edge_overlap': 139,
   'net_size': 197,
   'overlap_percent': 0.7055837563451777},
  'clustering_score': 0.503},
 'netmap': {'net_105_43582': {'edge_overlap': 26,
   'net_size': 110,
   'overlap_percent': 0.23636363636363636},
  'total_number_edges': 37050,
  'net_51_43266': {'edge_overlap': 46,
   'net_size': 76,
   'overlap_percent': 0.6052631578947368},
  'net_82_42088': {'edge_overlap': 125,
   'net_size': 197,
   'overlap_percent': 0.6345177664974619},
  'clustering_score': 1.0},
 'csnet': {'net_105_43582': {'edge_overlap': 0,
   'net_size': 110,
   'overlap_percent': 0.0},
  'total_number_edges': 32,
  'net_51_43266': {'edge_overlap': 0, 'net_size': 76, 'overlap_percent': 0.0},
  'net_82_42088': {'edge_overlap': 0, 

132
72
5


,source,target,n_cells,cell_barcode
0,hsa-miR-342,TRIM25,1000,cell1
1,USP18,TRIM21,1000,cell1
2,VAPB,TRIM21,1000,cell1
3,VGLL4,TRIM21,1000,cell1
4,WASF1,TRIM21,1000,cell1
...,...,...,...,...
74845,hsa-miR-661,ZNF507,1000,cell499.1
74846,ZW10,ZNF507,1000,cell499.1
74847,ADIPOQ,ACADVL,1000,cell499.1
74848,ADD1,ACADVL,1000,cell499.1
